# 10 · Separar las fechas antes de entrenar

Preparamos las fechas de entrenamiento y prueba de los filtros de momentum y reversión. El calendario se fija sobre los meses que tienen las diez variables disponibles después del calentamiento.

Con las salidas de **01 → 02 → 07 → 08** guardadas, selecciona `.venv` como kernel y ejecuta este cuaderno. La validación se prepara directamente con las características y etiquetas. El 09 es un diagnóstico descriptivo que puedes consultar por separado.

El [plan de validación temporal](../docs/plan_validacion_temporal.md#11-calendario-con-120-meses-de-calentamiento) documenta las decisiones. Todo el cálculo usa datos hasta diciembre de 1989.

## 1. Entradas y ejemplos utilizables

Las diez variables del 07 describen información anterior a cada resultado. Para preparar el z-score de cada mes usamos al menos **120 observaciones mensuales anteriores**, con los límites de recorte 1/99 calculados sobre ese pasado. Los meses de calentamiento sirven para estimar la escala de las variables y quedan fuera de los bloques.

Las etiquetas del 08 indican si una operación aislada habría superado al efectivo. Un ejemplo utilizable combina una señal activa, su etiqueta binaria y las diez variables listas. Los meses sin señal permanecen en el calendario con etiqueta ausente. La selección de variables para un modelo se hará dentro de su entrenamiento.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from IPython.display import display

from tfm.data import project_root, read_result, save_result
from tfm.screening import REGIME_MIN_HISTORY
from tfm.validation import (EMBARGO_MONTHS, MODEL_COLUMNS, N_BLOCKS, PURGE_MONTHS,
                            REGIME_CPCV_START, REGIME_RESULT_NAMES, VALIDATION_BLOCKS,
                            build_temporal_splits, prepare_validation_samples, summarize_validation)

ROOT = project_root()
caracteristicas, _ = read_result(ROOT, "monthly_features")
etiquetas, informe_etiquetas = read_result(ROOT, "labelled_signals")
ejemplos, informe_preparacion = prepare_validation_samples(
    caracteristicas, etiquetas, min_history=REGIME_MIN_HISTORY)
display(pd.DataFrame(informe_preparacion["sample_counts"]).rename(columns={
    "strategy": "Regla", "calendar_months": "Meses de calendario", "causal_feature_months": "Meses con preparación",
    "signal_months": "Meses con señal", "usable_examples": "Ejemplos utilizables", "positive": "Positivos", "negative": "Negativos"}))
print("Primera fecha común preparada:", informe_preparacion["first_causal_target"])
print("Diez predictores explícitos:", MODEL_COLUMNS)

## 2. Seis bloques y quince combinaciones

El tramo común va de agosto de 1937 a diciembre de 1989. Repartimos sus **629 meses en seis bloques** consecutivos, cinco de 105 meses y el último de 104. En cada una de las quince combinaciones reservamos dos bloques para comprobar las predicciones y partimos de los otros cuatro para entrenar.

La **validación cruzada combinatoria purgada (CPCV)** puede entrenar con bloques que vienen después del que valida. Sirve para comprobar la robustez del modelo al cambiar los bloques. El *walk-forward* del notebook 14 comprueba cómo funciona al aprender solo del pasado.

Cada mes aparece en cinco combinaciones de prueba. Es el mismo mes evaluado con cinco entrenamientos distintos. Las dos reglas usan las mismas fronteras y se distinguen por las señales activas de cada una.

In [ ]:
particiones, informe_calendario = build_temporal_splits(calendar_start=REGIME_CPCV_START)
resumen = summarize_validation(particiones, ejemplos, calendar_start=REGIME_CPCV_START)
display(pd.DataFrame(informe_calendario["blocks"]).rename(columns={
    "block_id": "Bloque", "first_target": "Primer objetivo", "last_target": "Último objetivo", "months": "Meses"}))
combinaciones = particiones[["split_id", "validation_block_1", "validation_block_2"]].drop_duplicates()
display(combinaciones.rename(columns={"split_id": "Partición", "validation_block_1": "Bloque de validación A",
                                     "validation_block_2": "Bloque de validación B"}).set_index("Partición"))
display(pd.Series({"Bloques": N_BLOCKS, "Bloques reservados por partición": VALIDATION_BLOCKS,
                   "Purga por frontera (meses)": PURGE_MONTHS, "Ventana de embargo posterior (meses)": EMBARGO_MONTHS},
                  name="Diseño fijado"))

## 3. Por qué dejamos separación entre fechas

Una etiqueta utiliza el resultado del mes objetivo. Evitamos que un ejemplo de entrenamiento toque el intervalo que estamos validando. Con nuestro horizonte mensual excluimos un mes antes y después de cada intervalo de validación, es una **purga conservadora**.

Después dejamos una ventana de **seis meses de embargo** desde el último mes objetivo de validación. Su primer mes ya coincide con la purga posterior, por lo que no se suma dos veces. Si los bloques de validación son adyacentes, se tratan como un solo intervalo. Las exclusiones se cuentan en meses calendario, aunque la regla no proponga operaciones durante alguno.

El gráfico muestra el papel de cada mes con las diez variables listas. Después contamos en cada bloque los meses en que cada regla emitió señal y tiene una etiqueta.

In [ ]:
colores = {"train": "#4C78A8", "validation": "#F58518", "purge": "#B279A2", "embargo": "#BDBDBD"}
rotulos = {"train": "Entrenamiento", "validation": "Validación", "purge": "Purga", "embargo": "Embargo"}
codigos = {nombre: i for i, nombre in enumerate(colores)}
matriz = particiones.pivot(index="split_id", columns="target_period", values="role")
imagen = matriz.apply(lambda column: column.map(codigos)).to_numpy(dtype=int)
fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(imagen, aspect="auto", interpolation="nearest", cmap=ListedColormap(list(colores.values())), vmin=-.5, vmax=3.5)
inicios = [bloque["first_target"] for bloque in informe_calendario["blocks"]]
marcas = [matriz.columns.get_loc(inicio) for inicio in inicios]
ax.set_xticks(marcas, inicios)
for marca in marcas[1:]:
    ax.axvline(marca - .5, color="white", alpha=.6, linewidth=.7)
ax.set_yticks(np.arange(len(matriz)), matriz.index)
ax.set(xlabel="Mes objetivo de inicio de cada bloque", ylabel="Partición",
       title="Quince particiones CPCV · agosto de 1937 a diciembre de 1989")
ax.legend(handles=[Patch(color=colores[role], label=rotulos[role]) for role in colores],
          loc="upper center", bbox_to_anchor=(.5, -.13), ncol=4)
plt.tight_layout()
plt.show()

## 4. Una partición y sus fronteras

Puedes cambiar el número de la partición. La tabla muestra el final de su último intervalo de prueba, el mes de purga posterior y los meses de embargo que caen dentro del calendario. Si la prueba termina en diciembre de 1989, la tabla se detiene ahí.

El primer mes posterior a un intervalo puede pertenecer a las dos ventanas. Su papel es «Purga» y, para sumar los papeles del calendario, se cuenta una sola vez.

In [ ]:
particion = 2
if particion not in set(particiones.split_id):
    raise ValueError("Escoge una partición entre 1 y 15.")
calendario = particiones.loc[particiones.split_id.eq(particion)].copy()
intervalos = informe_calendario["partitions"][particion - 1]["validation_intervals"]
print(f"Intervalos de validación de la partición {particion}")
display(pd.DataFrame(intervalos).rename(columns={"start": "Primer objetivo", "end": "Último objetivo"}))
columnas = ["target_period", "block_id", "role", "in_purge_window", "in_embargo_window"]
ultimo_objetivo = pd.Period(intervalos[-1]["end"], freq="M")
frontera = calendario.loc[calendario.target_period.between(
    str(ultimo_objetivo - 1), str(ultimo_objetivo + EMBARGO_MONTHS + 1)), columnas].copy()
frontera["role"] = frontera.role.map(rotulos)
display(frontera.rename(columns={"target_period": "Objetivo", "block_id": "Bloque", "role": "Papel",
    "in_purge_window": "En ventana de purga", "in_embargo_window": "En ventana de embargo"}).set_index("Objetivo"))

## 5. Cuántos ejemplos quedan para cada regla

Contamos por separado calendario, variables disponibles, señales y ejemplos utilizables. Los positivos y negativos corresponden a la etiqueta central. No se eliminan particiones ni se cambian las fronteras por obtener una distribución poco favorable.

Un entrenamiento con una sola clase no permite entrenar un clasificador convencional. Si falta una clase solo en validación, AUC no sería estimable. Son controles de cobertura, no puntuaciones de un modelo.

El final temporal de cada etiqueta debe coincidir con el mes objetivo. Esto no certifica cuándo se publicaron sus componentes: los dividendos reconstruidos y las revisiones de fuentes mantienen sus limitaciones. La disponibilidad efectiva para el futuro reentrenamiento cronológico deberá tratarse en esa etapa.

In [ ]:
columnas = ["strategy", "train_calendar_months", "train_feature_months", "train_signal_months", "train_examples",
            "train_positive", "train_negative", "validation_calendar_months", "validation_feature_months",
            "validation_signal_months", "validation_examples", "validation_positive", "validation_negative",
            "purge_calendar_months", "embargo_calendar_months", "embargo_window_months",
            "train_classifier_ready", "validation_auc_estimable"]
display(resumen.loc[resumen.split_id.eq(particion), columnas].set_index("strategy").T)
print("Rango de ejemplos entre las quince particiones:")
display(resumen.groupby("strategy")[["train_examples", "validation_examples"]].agg(["min", "max"]))
print("Particiones con ambas clases en entrenamiento:", int(resumen.train_classifier_ready.sum()), "de", len(resumen))
print("Particiones con ambas clases en validación:", int(resumen.validation_auc_estimable.sum()), "de", len(resumen))

## 6. Guardar las particiones

`validation_120_splits.parquet` guarda las **9.435 filas** del calendario de las quince combinaciones. `validation_120_summary.parquet` guarda sus treinta resúmenes, uno por regla y combinación. Los informes JSON recogen las fechas, el calentamiento y las exclusiones.

Si regeneras alguna entrada de 01, 02, 07 u 08, vuelve a ejecutar este notebook. Las dos tablas preparan el calendario para reproducir el modelo de referencia previsto en el nuevo diseño.

In [ ]:
informe_final = {**informe_calendario, **informe_preparacion,
    "scope": "development_partition_preparation_only",
    "parameters": informe_etiquetas["parameters"],
    "protocol": "docs/plan_validacion_temporal.md#11-calendario-con-120-meses-de-calentamiento",
    "references": ["Guía del tutor 28-11-2026, sublínea 1.B", "López de Prado (2018), capítulos 7 y 12"],
    "warnings": [
        "CPCV discovery can train on later blocks, it is not a real-time investment evaluation.",
        "Only 1927–1989, no model fitting, feature selection or post-1989 evaluation.",
        "Five validation appearances per month are not independent observations.",
        "Fixed causal features use past covariates, model fitting and outcome-based selection stay within training.",
        "Monthly Shiller means and reconstructed dividends, label end is not a certified publication date.",
        "Purge and embargo do not guarantee independence of all long-window indicators.",
    ],
}
guardados = []
for nombre, tabla in zip(REGIME_RESULT_NAMES, (particiones, resumen), strict=True):
    salida = save_result(ROOT, nombre, tabla, informe_final)
    guardados.append({"Tabla": nombre + ".parquet", "Filas": salida["rows"]})
display(pd.DataFrame(guardados))
print("Calendario CPCV y recuentos guardados en data/processed/.")